# 01 · Profiling the data


In [19]:
from silver import load_clean

df = load_clean()
print("Rows:", len(df))
df.head() 

Rows: 125053


,FACILITY,DEPARTMENT,PROVIDER_ID,APPOINTMENT_DATE,APPOINTMENT_TIME,PATIENT_ID,APPOINTMENT_TYPE,BOOKING_DATE,BOOKING_TIME,SHOW_CODE,...,booking_min_of_day,day_of_week,appt_month,is_telephone,is_after_hours,is_weekend,is_completed,is_noshow,lead_days,booking_lead_min
0,SCH,MED,67350046,12/04/09 00:00:00,12/30/99 17:30:00,12210741,Telephone Visit,12/04/09 00:00:00,12/30/99 15:05:00,Y,...,905,Friday,2009-12,True,True,False,True,False,0,145.0
1,SCH,MED,67350046,12/04/09 00:00:00,12/30/99 17:40:00,05491175,Telephone Visit,12/04/09 00:00:00,12/30/99 16:15:00,Y,...,975,Friday,2009-12,True,True,False,True,False,0,85.0
2,SCH,MED,67350046,12/04/09 00:00:00,12/30/99 17:50:00,15740599,Telephone Visit,12/04/09 00:00:00,12/30/99 16:43:00,Y,...,1003,Friday,2009-12,True,True,False,True,False,0,67.0
3,SCH,MED,67350046,12/04/09 00:00:00,12/30/99 18:00:00,80420422,Telephone Visit,12/04/09 00:00:00,12/30/99 16:53:00,Y,...,1013,Friday,2009-12,True,True,False,True,False,0,67.0
4,SCH,MED,67350046,12/04/09 00:00:00,12/30/99 18:10:00,86030202,Telephone Visit,12/04/09 00:00:00,12/30/99 16:57:00,Y,...,1017,Friday,2009-12,True,True,False,True,False,0,73.0


In [20]:
# The columns we actually use in the analysis: the IDs we group/join on, the
# cleaned dates/times, and the True/False flags created in silver.py.
columns_use = [
    "PROVIDER_ID", "PATIENT_ID",            # who: for per-provider and per-patient analysis
    "appt_date", "appt_hour", "appt_min_of_day",   # when (cleaned from the broken time column)
    "day_of_week", "appt_month",            # calendar groupings
    "booking_date", "booking_min_of_day",   # when it was booked
    "lead_days", "booking_lead_min",        # how far ahead it was booked
    "is_telephone", "is_after_hours", "is_weekend",  # what kind of appointment
    "is_completed", "is_noshow",            # did the patient show up
]
df[columns_use].head()

,PROVIDER_ID,PATIENT_ID,appt_date,appt_hour,appt_min_of_day,day_of_week,appt_month,booking_date,booking_min_of_day,lead_days,booking_lead_min,is_telephone,is_after_hours,is_weekend,is_completed,is_noshow
0,67350046,12210741,2009-12-04,17,1050,Friday,2009-12,2009-12-04,905,0,145.0,True,True,False,True,False
1,67350046,05491175,2009-12-04,17,1060,Friday,2009-12,2009-12-04,975,0,85.0,True,True,False,True,False
2,67350046,15740599,2009-12-04,17,1070,Friday,2009-12,2009-12-04,1003,0,67.0,True,True,False,True,False
3,67350046,80420422,2009-12-04,18,1080,Friday,2009-12,2009-12-04,1013,0,67.0,True,True,False,True,False
4,67350046,86030202,2009-12-04,18,1090,Friday,2009-12,2009-12-04,1017,0,73.0,True,True,False,True,False


## How many appointments, providers, patients, and what date range?

In [21]:
print("Date range:", df["appt_date"].min().date(), "to", df["appt_date"].max().date())
print("Providers:", df["PROVIDER_ID"].nunique())
print("Patients:", df["PATIENT_ID"].nunique())

Date range: 2009-12-01 to 2010-06-11
Providers: 150
Patients: 70588


## Everything is one clinic
`FACILITY` and `DEPARTMENT` only ever have one value, so there's no site-to-site comparison to make.
Also note the data is only ~6 months (Dec–Jun), so we can't really talk about seasonality.

In [22]:
print(df["FACILITY"].value_counts())
print(df["DEPARTMENT"].value_counts())

FACILITY
SCH    125053
Name: count, dtype: int64
DEPARTMENT
MED    125053
Name: count, dtype: int64


## Appointment types - only two, and telephone is called 'Telephone Visit'

In [23]:
df["APPOINTMENT_TYPE"].value_counts(normalize=True).mul(100).round(2)

APPOINTMENT_TYPE
In-Person Visit    89.43
Telephone Visit    10.57
Name: proportion, dtype: float64

## SHOW_CODE has THREE values, not two
The data dictionary writes "N/P" like it's one thing, but the column actually has `Y`, `N`, and `P`.
We treat both `N` and `P` as a no-show. There is **no** cancelled / rescheduled code anywhere.

In [24]:
df["SHOW_CODE"].value_counts()

SHOW_CODE
Y    111820
N     12573
P       660
Name: count, dtype: int64

## Did the time parsing work? Hours should look like a normal clinic day, not all midnight

In [25]:
df["appt_hour"].value_counts().sort_index()

appt_hour
5         9
7         1
8     13018
9     18176
10    18082
11    15655
12     1686
13    10453
14    16052
15    13370
16    10223
17     2100
18     2868
19     2412
20      947
21        1
Name: count, dtype: int64

## afterhours appointments are all same day
If we look at how far ahead appointments are booked, after hours appointments (telephone and in-person)
are basically always booked the same day. So this is a same  day, on demand service, not a planned diary.

In [26]:
# Average lead time (days booked in advance) for each group
df["group"] = df["is_telephone"].map({True: "Telephone", False: "In-Person"}) + " / " + \
              df["is_after_hours"].map({True: "After-Hours", False: "Daytime"})

summary = df.groupby("group").agg(
    appointments=("lead_days", "size"),
    pct_booked_same_day=("lead_days", lambda x: round((x == 0).mean() * 100)),
)
summary

,appointments,pct_booked_same_day
group,,
In-Person / After-Hours,2719,100
In-Person / Daytime,109115,35
Telephone / After-Hours,4383,100
Telephone / Daytime,8836,83


------------------------------------------------------------------------

**Q1** How many physicians do we need for the after hours telephone clinic?

In [68]:
after_hours_df = df[df["is_after_hours"]]
after_hours_breakdown = after_hours_df["is_telephone"].value_counts(normalize=True).mul(100).round(2)
print("After-hours appointments breakdown:")
print(f"Telephone: {after_hours_breakdown[True]}%")
print(f"In-Person: {after_hours_breakdown[False]}%")

After-hours appointments breakdown:
Telephone: 61.72%
In-Person: 38.28%


In [69]:
after_hours_telephone_df = df[df["is_after_hours"] & df["is_telephone"]]
after_hours_telephone_df

,FACILITY,DEPARTMENT,PROVIDER_ID,APPOINTMENT_DATE,APPOINTMENT_TIME,PATIENT_ID,APPOINTMENT_TYPE,BOOKING_DATE,BOOKING_TIME,SHOW_CODE,...,day_of_week,appt_month,is_telephone,is_after_hours,is_weekend,is_completed,is_noshow,lead_days,booking_lead_min,group
0,SCH,MED,67350046,12/04/09 00:00:00,12/30/99 17:30:00,12210741,Telephone Visit,12/04/09 00:00:00,12/30/99 15:05:00,Y,...,Friday,2009-12,True,True,False,True,False,0,145.0,Telephone / After-Hours
1,SCH,MED,67350046,12/04/09 00:00:00,12/30/99 17:40:00,05491175,Telephone Visit,12/04/09 00:00:00,12/30/99 16:15:00,Y,...,Friday,2009-12,True,True,False,True,False,0,85.0,Telephone / After-Hours
2,SCH,MED,67350046,12/04/09 00:00:00,12/30/99 17:50:00,15740599,Telephone Visit,12/04/09 00:00:00,12/30/99 16:43:00,Y,...,Friday,2009-12,True,True,False,True,False,0,67.0,Telephone / After-Hours
3,SCH,MED,67350046,12/04/09 00:00:00,12/30/99 18:00:00,80420422,Telephone Visit,12/04/09 00:00:00,12/30/99 16:53:00,Y,...,Friday,2009-12,True,True,False,True,False,0,67.0,Telephone / After-Hours
4,SCH,MED,67350046,12/04/09 00:00:00,12/30/99 18:10:00,86030202,Telephone Visit,12/04/09 00:00:00,12/30/99 16:57:00,Y,...,Friday,2009-12,True,True,False,True,False,0,73.0,Telephone / After-Hours
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124509,SCH,MED,80069968,04/03/10 00:00:00,12/30/99 18:40:00,82350469,Telephone Visit,04/03/10 00:00:00,12/30/99 17:07:00,Y,...,Saturday,2010-04,True,True,True,True,False,0,93.0,Telephone / After-Hours
124510,SCH,MED,80069968,04/03/10 00:00:00,12/30/99 18:50:00,28071143,Telephone Visit,04/03/10 00:00:00,12/30/99 17:35:00,Y,...,Saturday,2010-04,True,True,True,True,False,0,75.0,Telephone / After-Hours
124512,SCH,MED,80069968,04/03/10 00:00:00,12/30/99 19:30:00,23380748,Telephone Visit,04/03/10 00:00:00,12/30/99 17:45:00,Y,...,Saturday,2010-04,True,True,True,True,False,0,105.0,Telephone / After-Hours
124513,SCH,MED,80069968,04/03/10 00:00:00,12/30/99 19:40:00,10341282,Telephone Visit,04/03/10 00:00:00,12/30/99 18:41:00,Y,...,Saturday,2010-04,True,True,True,True,False,0,59.0,Telephone / After-Hours
